# ENEM 2024 · Sprint 5 — Classificação do risco de zerar a redação

**Alvo:** `y_zerada` — 1 quando a redação foi entregue mas **não** foi corrigida sem problema
(anulada, em branco, fuga ao tema, texto insuficiente). São ~165 mil casos em 3,17 milhões: **um
problema desbalanceado**, e é isso que muda tudo em relação à Sprint 4.

Este notebook **não** usa regressão linear. Ele responde a outra pergunta e, principalmente, usa
**outra prática de avaliação**: em vez de uma tabela de R², a escolha do modelo e do ponto de corte é
feita **olhando curvas**.

O que tem aqui:

1. por que **acurácia** mente num problema desbalanceado;
2. duas formas de tratar o desbalanceamento — `class_weight="balanced"` e **ajuste do limiar** —
   comparadas nas mesmas 5 rodadas;
3. **curva ROC** e **curva precisão–recall**, para escolher o modelo;
4. **curva de calibração**, para saber se a probabilidade prevista quer dizer alguma coisa;
5. **precisão/recall × limiar** e **curva de ganho acumulado**, para escolher o ponto de corte;
6. **equidade**: o modelo erra igual entre tipos de escola e cor/raça?

Comparamos um modelo **linear nos parâmetros** (regressão logística) com um **não linear**
(`HistGradientBoostingClassifier`, árvores com boosting, já incluso no scikit-learn — sem instalar nada).

⚠️ Como na Sprint 4, **tudo roda só no conjunto de treino**. O teste fica guardado.

## 0 · Setup

Mesma pasta e mesmas convenções da Sprint 4. Precisa de **scikit-learn ≥ 1.4** (o `class_weight` do
`HistGradientBoostingClassifier` entrou nessa versão); o Colab já vem com uma mais nova.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from sklearn.calibration import calibration_curve
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (average_precision_score, confusion_matrix,
                             precision_recall_curve, roc_auc_score, roc_curve)
from sklearn.model_selection import StratifiedKFold, cross_validate, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

print("scikit-learn", sklearn.__version__)
assert tuple(int(p) for p in sklearn.__version__.split(".")[:2]) >= (1, 4), "precisa de sklearn >= 1.4"

PASTA   = os.environ.get("ENEM_OUT", "data_processed")
ARQ     = os.path.join(PASTA, "enem2024_features_ordinal.parquet")
ALVO    = "y_zerada"          # troque por "y_alto" ou "y_fez_redacao" e o notebook inteiro roda igual
SEED    = 42
AMOSTRA = 300_000             # a logistica em 5 rodadas fica lenta acima disso (None = tudo)

# estratificado: com 5% de positivos, um KFold comum pode deixar uma rodada com poucos casos
CV5 = StratifiedKFold(5, shuffle=True, random_state=0)

pd.set_option("display.width", 160)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": .25})

AZUL, LARANJA, VERDE, CINZA = "#06406F", "#c2410c", "#15803d", "#9ca3af"

## 1 · Leitura e o tamanho do desbalanceamento

Nota: `y_zerada` só existe para quem **entregou** a redação (`y_fez_redacao == 1`). Quem faltou tem
`NaN` e fica de fora — essa é outra pergunta, a do alvo `y_fez_redacao`.

In [ ]:
FEATURES = pd.read_csv(os.path.join(PASTA, "lista_features.csv"))["feature"].tolist()
CAT = ["cor_raca", "regiao", "uf"]
TE  = ["te_municipio", "te_uf"]                          # vazam na CV (Sprint 4, secao 3): ficam de fora
NUM = [c for c in FEATURES if c not in CAT]
NUM_SEM_TE = [c for c in NUM if c not in TE]

base   = pd.read_parquet(ARQ, columns=["split", ALVO] + FEATURES)
treino = base[(base["split"] == "treino") & base[ALVO].notna()]
df = treino if AMOSTRA is None else treino.sample(min(AMOSTRA, len(treino)), random_state=SEED)
df = df.reset_index(drop=True)
del base, treino

X = df[FEATURES].copy()
X[NUM] = X[NUM].astype("float64")
X[CAT] = X[CAT].astype(str)
y = df[ALVO].astype("int8")

prev = y.mean()
print(f"linhas: {len(X):,}")
print(f"positivos ({ALVO} = 1): {y.sum():,}  ->  {prev:.2%}")
print(f"negativos:              {(1 - y).sum():,}  ->  {1 - prev:.2%}")
print(f"\nrazao negativo:positivo = {(1 - prev) / prev:.1f} : 1")

**Por que isso é um problema.** Um modelo que responde **sempre "não zera"** acerta a fração acima
de negativos. Se os positivos são 5%, esse modelo burro tem **95% de acurácia** e é completamente
inútil: ele nunca encontra ninguém em risco. É a armadilha da acurácia em base desbalanceada.

> ✍️ **Leitura do grupo:** o alvo tem ___% de positivos, ou seja ___ negativos para cada positivo.

## 2 · O piso: acurácia mente, as outras métricas não

Comparamos o **chute** (prever sempre a classe majoritária) com uma **regressão logística**, nas mesmas
5 rodadas estratificadas. As métricas:

| Métrica | O que mede | Por que está aqui |
|---|---|---|
| **Acurácia** | % de acertos | só para **mostrar que engana** |
| **ROC AUC** | chance de ordenar um positivo acima de um negativo | não depende do limiar nem da prevalência |
| **PR AUC** | área da curva precisão–recall (`average_precision`) | **a métrica certa** quando a classe de interesse é rara |
| **Recall** | dos que zeraram, quantos o modelo pegou | o custo de errar aqui é deixar passar |
| **Precisão** | dos que o modelo apontou, quantos zeraram mesmo | o custo de errar aqui é alarme falso |

O pré-processamento muda em relação à Sprint 4: a logística **precisa** das numéricas padronizadas
(`StandardScaler`), senão a renda em reais domina a escala e o `lbfgs` não converge direito.

In [ ]:
CAT_OHE = ["cor_raca", "uf"]          # regiao fica fora: e funcao da UF (Sprint 4, secao 2)
METRICAS = {"ROC AUC": "roc_auc", "PR AUC": "average_precision",
            "Recall": "recall", "Precisao": "precision", "Acuracia": "accuracy"}

def preparar(numericas=NUM_SEM_TE, onehot=CAT_OHE):
    # padroniza as numericas e faz one-hot DENTRO do pipeline, para nao vazar entre as rodadas
    return ColumnTransformer([
        ("num", StandardScaler(), list(numericas)),
        ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), list(onehot)),
    ])

def logistica(class_weight=None):
    return Pipeline([("prep", preparar()),
                     ("modelo", LogisticRegression(max_iter=1000, class_weight=class_weight,
                                                   random_state=SEED))])

def arvores(class_weight=None):
    return Pipeline([("prep", preparar()),
                     ("modelo", HistGradientBoostingClassifier(class_weight=class_weight,
                                                               random_state=SEED))])

def avaliar(nome, modelo):
    # roda a CV estratificada de 5 partes e devolve media e desvio de cada metrica
    r = cross_validate(modelo, X, y, cv=CV5, scoring=METRICAS, n_jobs=-1)
    linha = {"modelo": nome}
    for m in METRICAS:
        v = r[f"test_{m}"]
        linha[m], linha[f"{m} dp"] = v.mean(), v.std()
    return linha

def tabela(linhas):
    t = pd.DataFrame(linhas).set_index("modelo")
    return pd.DataFrame({m: t[m].map("{:.3f}".format) + " \u00b1 " + t[f"{m} dp"].map("{:.3f}".format)
                         for m in METRICAS})

res = [avaliar("Chute: sempre a classe majoritaria", DummyClassifier(strategy="most_frequent")),
       avaliar("Logistica", logistica())]
tabela(res)

**Como ler:** olhe a coluna **Acurácia** do chute e compare com o **ROC AUC** dele (0,5, que é o
mesmo que jogar moeda) e com o **Recall** (0, ele não acha ninguém). É a demonstração de que acurácia
alta não quer dizer modelo útil.

Repare também no **recall da logística sem tratamento**: ele costuma vir baixo. O modelo otimiza o erro
total, e com 5% de positivos o jeito mais fácil de errar pouco é quase nunca dizer "zera".

> ✍️ **Leitura do grupo:** o chute teve ___ de acurácia e ___ de recall. A logística sem tratamento
> teve ROC AUC ___ e recall ___.

## 3 · Tratando o desbalanceamento

Duas práticas, nas **mesmas** 5 rodadas:

1. **`class_weight="balanced"`** — dá a cada classe um peso inversamente proporcional à sua frequência,
   então cada positivo "vale" ~19 negativos no cálculo do erro. Não inventa nem joga fora linha nenhuma.
2. **Modelo não linear** (`HistGradientBoosting`), com e sem o mesmo peso, para separar o que é ganho de
   **tratar o desbalanceamento** do que é ganho de **mudar o modelo**.

> **Por que não SMOTE / undersampling.** SMOTE inventa linhas sintéticas interpolando vizinhos — num
> espaço cheio de variável binária e ordinal (as 23 questões) isso cria gente que não existe, e exige
> uma dependência a mais (`imblearn`). Undersampling joga fora 95% dos dados. O `class_weight` consegue
> o mesmo efeito sem nenhuma das duas desvantagens, e o **ajuste do limiar** (seção 6) é o que de fato
> resolve. Se o grupo quiser testar SMOTE depois, vale como experimento — mas não como ponto de partida.

In [ ]:
res += [avaliar("Logistica + class_weight=balanced", logistica("balanced")),
        avaliar("Arvores (HistGradientBoosting)", arvores()),
        avaliar("Arvores + class_weight=balanced", arvores("balanced"))]
tabela(res)

**Como ler:** `balanced` quase sempre **sobe muito o recall e derruba a precisão** — o modelo passa
a apontar muito mais gente. Olhe o **PR AUC**: se ele quase não muda, o peso **não criou informação
nova**, só deslocou o modelo ao longo da mesma curva de trade-off. Isso é esperado, e é a razão de a
seção 6 existir: o que escolhe o ponto do trade-off é o **limiar**, não o peso.

A comparação que vale para escolher o modelo é **PR AUC e ROC AUC**, com o desvio das 5 rodadas ao lado.

> ✍️ **Leitura do grupo:** `balanced` mudou o recall de ___ para ___ e a precisão de ___ para ___.
> O PR AUC foi de ___ para ___. O melhor PR AUC foi do modelo ___.

## 4 · Curva ROC e curva precisão–recall

Aqui entra a prática que dá nome ao notebook: em vez de comparar números soltos, **desenhamos o
trade-off inteiro**. Cada ponto da curva é um limiar diferente.

As probabilidades são **fora da amostra** (`cross_val_predict` com `method="predict_proba"`): cada linha
é prevista pela rodada em que ela estava na validação.

In [ ]:
MODELOS = {"Logistica": logistica(),
           "Logistica balanced": logistica("balanced"),
           "Arvores": arvores()}

probas = {nome: cross_val_predict(m, X, y, cv=CV5, method="predict_proba", n_jobs=-1)[:, 1]
          for nome, m in MODELOS.items()}

fig, ax = plt.subplots(1, 2, figsize=(11.5, 4.4))
for (nome, p), cor in zip(probas.items(), [AZUL, LARANJA, VERDE]):
    fpr, tpr, _ = roc_curve(y, p)
    ax[0].plot(fpr, tpr, color=cor, lw=1.6, label=f"{nome} (AUC {roc_auc_score(y, p):.3f})")
    prec, rec, _ = precision_recall_curve(y, p)
    ax[1].plot(rec, prec, color=cor, lw=1.6, label=f"{nome} (AP {average_precision_score(y, p):.3f})")

ax[0].plot([0, 1], [0, 1], "--", color=CINZA, lw=1, label="moeda (AUC 0,500)")
ax[0].set_xlabel("falso positivo (alarme falso)"); ax[0].set_ylabel("verdadeiro positivo (recall)")
ax[0].set_title("ROC"); ax[0].legend(fontsize=8, loc="lower right")

ax[1].axhline(prev, ls="--", color=CINZA, lw=1, label=f"chute ({prev:.3f})")
ax[1].set_xlabel("recall"); ax[1].set_ylabel("precisao")
ax[1].set_title("Precisao x Recall"); ax[1].legend(fontsize=8, loc="upper right")
plt.tight_layout(); plt.show()

**Como ler as duas curvas:**

- **ROC:** quanto mais a curva se afasta da diagonal **para cima e para a esquerda**, melhor. O problema
  da ROC em base desbalanceada é que o eixo x (falso positivo) é dividido pelos **negativos**, que são
  muitos — então até um modelo que gera muito alarme falso parece bem.
- **Precisão–recall:** é a honesta aqui. A linha tracejada é a **prevalência**, o que o chute aleatório
  consegue. A distância da curva até essa linha é o ganho real do modelo.

Repare que **a logística com e sem `balanced` desenham praticamente a mesma curva PR** — porque são o
mesmo modelo, só com o corte em lugar diferente. As curvas mostram isso de um jeito que a tabela não
mostra.

> ✍️ **Leitura do grupo:** a prevalência é ___, e o melhor modelo chega a ___ de precisão quando o
> recall é 0,5. Escolhemos o modelo ___ porque ___.

## 5 · Calibração: a probabilidade quer dizer alguma coisa?

Se o modelo diz "30% de chance de zerar" para mil pessoas, **300 deveriam zerar**. Um modelo pode
ordenar bem (AUC alto) e ainda assim dar probabilidades erradas — e aí o número não pode ser mostrado
para ninguém.

In [ ]:
fig, ax = plt.subplots(figsize=(5.4, 4.6))
for (nome, p), cor in zip(probas.items(), [AZUL, LARANJA, VERDE]):
    real, previsto = calibration_curve(y, p, n_bins=15, strategy="quantile")
    ax.plot(previsto, real, "o-", ms=4, lw=1.4, color=cor, label=nome)
ax.plot([0, 1], [0, 1], "--", color=CINZA, lw=1, label="calibracao perfeita")
lim = max(max(p.max() for p in probas.values()), prev * 3)
ax.set_xlim(0, lim); ax.set_ylim(0, lim)
ax.set_xlabel("probabilidade prevista"); ax.set_ylabel("fracao que de fato zerou")
ax.set_title("Calibracao (15 grupos de mesmo tamanho)")
ax.legend(fontsize=8); plt.tight_layout(); plt.show()

**Como ler:** em cima da diagonal = calibrado. **Acima** da diagonal = o modelo **subestima** o
risco; **abaixo** = superestima.

A `balanced` deve aparecer **bem abaixo da diagonal**: ao dar peso 19× aos positivos, ela infla a
probabilidade de todo mundo. Esse é o **preço escondido** do `class_weight` — ela melhora o recall e
**estraga a calibração**. É mais um argumento para resolver o desbalanceamento pelo limiar: a logística
sem peso continua calibrada, e o limiar move o corte sem mexer na probabilidade.

> ✍️ **Leitura do grupo:** o modelo mais bem calibrado foi ___. A `balanced` ficou ___ da diagonal,
> o que quer dizer ___.

## 6 · Escolhendo o limiar

`predict()` usa 0,5 por padrão. Num problema com 5% de positivos, **0,5 é quase sempre a escolha errada**
— quase ninguém passa desse corte. O limiar é um botão que o grupo gira conforme o custo do erro:

- **limiar baixo** → acha quase todos os casos (recall alto), com muito alarme falso (precisão baixa);
- **limiar alto** → quase todo apontamento está certo (precisão alta), mas deixa muita gente passar.

Como não existe limiar "certo" sem uma decisão, mostramos a curva inteira e marcamos **duas** opções:
o que **maximiza o F1** (equilíbrio) e o que garante **recall de 70%** (se a prioridade for não deixar
passar).

In [ ]:
MODELO_ESCOLHIDO = "Logistica"        # troque aqui depois de olhar as curvas da secao 4
p = probas[MODELO_ESCOLHIDO]

prec, rec, lim = precision_recall_curve(y, p)
prec, rec = prec[:-1], rec[:-1]                       # precision_recall_curve devolve 1 ponto a mais
f1 = np.divide(2 * prec * rec, prec + rec, out=np.zeros_like(prec), where=(prec + rec) > 0)

lim_f1  = lim[f1.argmax()]
alvo_rec = 0.70
lim_rec = lim[rec >= alvo_rec].max() if (rec >= alvo_rec).any() else lim.min()

fig, ax = plt.subplots(figsize=(7.5, 4.2))
ax.plot(lim, prec, color=AZUL, lw=1.5, label="precisao")
ax.plot(lim, rec, color=LARANJA, lw=1.5, label="recall")
ax.plot(lim, f1, color=VERDE, lw=1.5, label="F1")
ax.axvline(0.5, ls=":", color=CINZA, lw=1.2, label="padrao do predict() = 0,5")
ax.axvline(lim_f1, ls="--", color=VERDE, lw=1.2, label=f"F1 maximo = {lim_f1:.3f}")
ax.axvline(lim_rec, ls="--", color=LARANJA, lw=1.2, label=f"recall {alvo_rec:.0%} = {lim_rec:.3f}")
ax.set_xlabel("limiar"); ax.set_ylabel("metrica"); ax.set_xlim(0, min(1, lim.max()))
ax.set_title(f"{MODELO_ESCOLHIDO}: as tres metricas ao longo do limiar")
ax.legend(fontsize=8); plt.tight_layout(); plt.show()

def matriz(limiar, nome):
    pred = (p >= limiar).astype(int)
    vn, fp, fn, vp = confusion_matrix(y, pred).ravel()
    return {"limiar": round(float(limiar), 3), "apontados": vp + fp,
            "acertou (VP)": vp, "alarme falso (FP)": fp, "deixou passar (FN)": fn,
            "precisao": round(vp / max(vp + fp, 1), 3), "recall": round(vp / max(vp + fn, 1), 3)}

pd.DataFrame([matriz(0.5, "padrao"), matriz(lim_f1, "F1"), matriz(lim_rec, "recall 70%")],
             index=["padrao 0,5", "F1 maximo", f"recall {alvo_rec:.0%}"])

**Como ler a tabela:** cada linha é a **mesma** lista de probabilidades, cortada em lugar diferente.
Olhe `apontados`: é quanta gente o grupo teria de olhar na prática. E `deixou passar`: é quanta gente em
risco ficaria de fora.

Nenhuma linha é "a certa" — a escolha depende do que o erro custa. Para uma política de **apoio** (oferecer
oficina de redação), alarme falso é barato e deixar passar é caro: limiar baixo. Para qualquer coisa
**restritiva**, o raciocínio se inverte — e aí vale lembrar que este modelo usa perfil socioeconômico, e
usar isso para restringir alguém é indefensável.

> ✍️ **Leitura do grupo:** escolhemos o limiar ___ porque ___. Com ele, o modelo aponta ___ pessoas,
> acerta ___ e deixa passar ___.

## 7 · Curva de ganho acumulado: vale a pena priorizar?

A pergunta prática: **se desse para acompanhar só uma parte dos inscritos, quantos casos de risco o
modelo entregaria?** A curva ordena todo mundo pela probabilidade e mostra quanto do total de positivos
já foi coberto.

In [ ]:
ordem = np.argsort(-p)
ganho = np.cumsum(y.to_numpy()[ordem]) / y.sum()
frac  = np.arange(1, len(y) + 1) / len(y)

fig, ax = plt.subplots(figsize=(5.6, 4.4))
ax.plot(frac, ganho, color=AZUL, lw=1.8, label=MODELO_ESCOLHIDO)
ax.plot([0, 1], [0, 1], "--", color=CINZA, lw=1, label="sem modelo (aleatorio)")
ax.plot([0, prev, 1], [0, 1, 1], ":", color=VERDE, lw=1.2, label="modelo perfeito")
for marco in (0.10, 0.20):
    ax.annotate(f"{ganho[int(marco * len(y)) - 1]:.0%} dos casos\nnos {marco:.0%} piores",
                xy=(marco, ganho[int(marco * len(y)) - 1]), xytext=(marco + .12, ganho[int(marco * len(y)) - 1] - .18),
                fontsize=8, arrowprops=dict(arrowstyle="->", color=LARANJA, lw=1))
ax.set_xlabel("fracao dos inscritos acompanhada (do mais arriscado para o menos)")
ax.set_ylabel("fracao dos casos de risco encontrada")
ax.set_title("Ganho acumulado"); ax.legend(fontsize=8, loc="lower right")
plt.tight_layout(); plt.show()

**Como ler:** a distância entre a curva azul e a diagonal é o valor do modelo. "Nos 10% mais
arriscados estão X% de quem zera" é a frase que um gestor entende — e é mais útil que qualquer AUC.

> ✍️ **Leitura do grupo:** acompanhando os 10% mais arriscados, o modelo encontra ___% dos casos
> (sem modelo seriam 10%).

## 8 · Equidade: o modelo erra igual para todos?

Mesma preocupação da Sprint 4, adaptada para classificação. No limiar escolhido, para cada grupo:

- **taxa real** — quanto o grupo de fato zera;
- **taxa apontada** — quanto o modelo sinaliza (se for muito maior que a real, o grupo leva alarme falso
  em excesso);
- **recall** — dos que zeraram no grupo, quantos o modelo pegou;
- **precisão** — dos apontados no grupo, quantos zeraram mesmo.

Recall muito diferente entre grupos quer dizer que a **proteção** que o modelo oferece não é a mesma
para todo mundo.

In [ ]:
LIMIAR = lim_f1                      # troque pelo limiar decidido na secao 6
pred = (p >= LIMIAR).astype(int)

tipo_escola = np.select([X["escola_privada"] == 1, X["escola_mista"] == 1, X["escola_publica"] == 1],
                        ["privada", "mista", "publica"], default="sem EM / outra")

def equidade(grupo, nome):
    aux = pd.DataFrame({"grupo": grupo, "y": y.to_numpy(), "pred": pred})
    def linha(g):
        vp = int(((g.y == 1) & (g.pred == 1)).sum()); fp = int(((g.y == 0) & (g.pred == 1)).sum())
        fn = int(((g.y == 1) & (g.pred == 0)).sum())
        return pd.Series({"linhas": len(g), "taxa real": g.y.mean(), "taxa apontada": g.pred.mean(),
                          "recall": vp / max(vp + fn, 1), "precisao": vp / max(vp + fp, 1)})
    t = aux.groupby("grupo").apply(linha, include_groups=False).sort_values("linhas", ascending=False)
    t["linhas"] = t["linhas"].astype(int)
    return t.round(3).rename_axis(nome)

display(equidade(tipo_escola, "tipo de escola"))
display(equidade(X["cor_raca"].to_numpy(), "cor/raca"))

**Como ler:** compare **taxa real** com **taxa apontada** em cada linha. Se um grupo tem taxa real
de 4% e apontada de 15%, o modelo está sinalizando esse grupo três vezes mais do que o problema aparece
nele. Compare também o **recall** entre linhas: um grupo com recall muito menor é um grupo que o modelo
**protege menos**.

`cor_raca` está no modelo **para medir isso**, não como explicação da nota.

> ✍️ **Leitura do grupo:** o maior recall foi em ___ (___) e o menor em ___ (___). O grupo com maior
> diferença entre taxa real e apontada foi ___.

## 9 · Resumo da Sprint 5 e próximos passos

| | |
|---|---|
| **Alvo** | `y_zerada`, ___% de positivos (___ negativos por positivo) |
| **Chute** | acurácia ___ e recall 0,000 — a demonstração de que acurácia não serve aqui |
| **Melhor modelo** | ___ , PR AUC ___ ± ___ e ROC AUC ___ ± ___ |
| **Desbalanceamento** | `class_weight="balanced"` levou o recall de ___ para ___, sem mudar o PR AUC, e **piorou a calibração** |
| **Limiar escolhido** | ___ (critério: ___), apontando ___ pessoas, com precisão ___ e recall ___ |
| **Ganho** | nos 10% mais arriscados estão ___% dos casos |
| **Equidade** | ___ |

**O que este notebook mostra de diferente da Sprint 4:** lá a escolha era uma tabela de R²; aqui a
escolha é feita **em cima de curvas** — ROC e PR para escolher o modelo, calibração para saber se a
probabilidade é usável, precisão/recall × limiar e ganho acumulado para escolher o corte. Num problema
desbalanceado, nenhum número isolado (e muito menos a acurácia) sustenta a decisão.

**Próximos passos:**

- repetir com `y_alto` (nota ≥ 700) e `y_fez_redacao` (abstenção) — a única mudança é a constante `ALVO`;
- ajustar hiperparâmetros do `HistGradientBoosting` nas mesmas 5 rodadas;
- se a calibração do modelo escolhido ficar ruim, testar `CalibratedClassifierCV`;
- só depois de tudo decidido, avaliar **uma vez** no conjunto de teste.